In [10]:
import cupy as cp
import cupyx
from typing import Any

Class implementation

In [11]:
class MaxPool2D:
    """2D Max Pooling layer with support for forward and backward passes.

    Parameters
    ----------
    kernel_size : int or tuple[int, int]
        Size of the pooling window.
    stride : int or tuple[int, int] or None
        Stride of the pooling window. Defaults to kernel_size.
    padding : int or tuple[int, int]
        Zero-padding added to both sides of the input.
    """

    def __init__(self, kernel_size=2, stride=None, padding=0):
        self.kernel_size = (kernel_size, kernel_size) if isinstance(kernel_size, int) else tuple[Any, ...](kernel_size)
        self.stride = self.kernel_size if stride is None else (stride, stride) if isinstance(stride, int) else tuple[Any, ...](stride)
        self.padding = (padding, padding) if isinstance(padding, int) else tuple[Any, ...](padding)

        self._input_shape = None
        self._max_indices = None

    def forward(self, x: cp.ndarray) -> cp.ndarray:
        """
        Parameters
        ----------
        x : cp.ndarray of shape (N, C, H, W)

        Returns
        -------
        cp.ndarray of shape (N, C, H_out, W_out)
        """
        assert x.ndim == 4, f"Expected 4D input (N, C, H, W), got {x.ndim}D"
        self._input_shape = x.shape
        n, c, h, w = x.shape
        kh, kw = self.kernel_size
        sh, sw = self.stride
        ph, pw = self.padding

        if ph > 0 or pw > 0:
            x = cp.pad(x, ((0, 0), (0, 0), (ph, ph), (pw, pw)),
                        mode="constant", constant_values=-cp.inf)

        h_out = (h + 2 * ph - kh) // sh + 1
        w_out = (w + 2 * pw - kw) // sw + 1

        # Build strided view of all pooling windows: (N, C, H_out, W_out, kh, kw)
        strides = x.strides
        windows = cp.lib.stride_tricks.as_strided(
            x,
            shape=(n, c, h_out, w_out, kh, kw),
            strides=(strides[0], strides[1],
                     strides[2] * sh, strides[3] * sw,
                     strides[2], strides[3]),
        )

        out = windows.reshape(n, c, h_out, w_out, -1).max(axis=-1)

        flat_idx = windows.reshape(n, c, h_out, w_out, -1).argmax(axis=-1)
        self._max_indices = flat_idx

        return out

    def backward(self, grad_output: cp.ndarray) -> cp.ndarray:
        """
        Parameters
        ----------
        grad_output : cp.ndarray of shape (N, C, H_out, W_out)

        Returns
        -------
        grad_input : cp.ndarray of shape (N, C, H, W) — same shape as the forward input.
        """
        assert self._input_shape is not None, "Call forward() before backward()"
        n, c, h, w = self._input_shape
        kh, kw = self.kernel_size
        sh, sw = self.stride
        ph, pw = self.padding

        h_padded = h + 2 * ph
        w_padded = w + 2 * pw
        grad_padded = cp.zeros((n, c, h_padded, w_padded), dtype=grad_output.dtype)

        h_out, w_out = grad_output.shape[2], grad_output.shape[3]

        row_offsets = self._max_indices // kw
        col_offsets = self._max_indices % kw

        nn = cp.arange(n)[:, None, None, None]
        cc = cp.arange(c)[None, :, None, None]
        hh = cp.arange(h_out)[None, None, :, None]
        ww = cp.arange(w_out)[None, None, None, :]

        h_idx = hh * sh + row_offsets
        w_idx = ww * sw + col_offsets

        cupyx.scatter_add(grad_padded, (nn, cc, h_idx, w_idx), grad_output)

        if ph > 0 or pw > 0:
            return grad_padded[:, :, ph:ph + h, pw:pw + w]
        return grad_padded

Tests

In [12]:
cp.random.seed(42)

# --- Forward pass test ---
pool = MaxPool2D(kernel_size=2, stride=2, padding=0)

x = cp.array([[
    [[1, 2, 3, 4],
     [5, 6, 7, 8],
     [9, 10, 11, 12],
     [13, 14, 15, 16]],
]], dtype=cp.float32)  # (1, 1, 4, 4)

out = pool.forward(x)
expected = cp.array([[[[6, 8], [14, 16]]]], dtype=cp.float32)
assert cp.array_equal(out, expected), f"Forward mismatch:\n{out}\nvs expected:\n{expected}"
print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")
print(f"Output:\n{out}\n")

# --- Backward pass test ---
grad_out = cp.ones_like(out)
grad_in = pool.backward(grad_out)
print(f"Grad input shape: {grad_in.shape}")
print(f"Grad input:\n{grad_in}\n")

assert grad_in.shape == x.shape
assert float(grad_in[0, 0, 1, 1]) == 1.0, "Max element at (1,1) should receive gradient"
assert float(grad_in[0, 0, 0, 0]) == 0.0, "Non-max element at (0,0) should be zero"

# --- Larger random batch ---
batch = cp.random.randn(8, 3, 32, 32, dtype=cp.float32)
pool_large = MaxPool2D(kernel_size=2, stride=2)
out_large = pool_large.forward(batch)
assert out_large.shape == (8, 3, 16, 16), f"Unexpected shape: {out_large.shape}"

grad_large = pool_large.backward(cp.ones_like(out_large))
assert grad_large.shape == batch.shape

# --- Padding test ---
pool_pad = MaxPool2D(kernel_size=3, stride=1, padding=1)
out_pad = pool_pad.forward(x)
assert out_pad.shape == (1, 1, 4, 4), f"Padded output shape mismatch: {out_pad.shape}"
grad_pad = pool_pad.backward(cp.ones_like(out_pad))
assert grad_pad.shape == x.shape

print("All tests passed.")

Input shape:  (1, 1, 4, 4)
Output shape: (1, 1, 2, 2)
Output:
[[[[ 6.  8.]
   [14. 16.]]]]

Grad input shape: (1, 1, 4, 4)
Grad input:
[[[[0. 0. 0. 0.]
   [0. 1. 0. 1.]
   [0. 0. 0. 0.]
   [0. 1. 0. 1.]]]]

All tests passed.
